### MTD Performance vs Previous MTD

In [0]:
%sql
CREATE OR REPLACE VIEW `03_gold_catalog`.gold_kpi.v_mtd_performance AS
SELECT
    s.store_name,
    ds.manager_name,
    SUM(fi.invoice_amount) AS mtd_revenue,
    COUNT(DISTINCT fo.order_id) AS mtd_completed_orders,
    DATE_TRUNC('month', fi.invoice_ts) AS month_start
FROM `03_gold_catalog`.gold.fact_invoice_dim fi
JOIN `03_gold_catalog`.gold.fact_order_dim fo ON fi.order_id = fo.order_id
JOIN `03_gold_catalog`.gold.dim_store s ON fi.store_key = s.store_key
JOIN `03_gold_catalog`.gold.dim_store ds ON fo.store_key = ds.store_key
WHERE fi.invoice_ts >= DATE_TRUNC('month', CURRENT_DATE())
GROUP BY s.store_name, ds.manager_name, DATE_TRUNC('month', fi.invoice_ts);

In [0]:
%sql
select * from `03_gold_catalog`.gold_kpi.v_mtd_performance

### Average Days in Shop

In [0]:
%sql
CREATE OR REPLACE VIEW `03_gold_catalog`.gold_kpi.v_avg_days_in_shop AS
SELECT
    s.store_name,
    fo.service_type,
    COUNT(*) AS total_orders,
    ROUND(AVG(DATEDIFF(fo.vehicle_out_datetime, fo.vehicle_in_datetime)), 2) AS avg_days_in_shop
    -- AVG(fo.days_in_shop) AS avg_days_in_shop
FROM `03_gold_catalog`.gold.fact_order_dim fo
JOIN `03_gold_catalog`.gold.dim_store s ON fo.store_key = s.store_key
GROUP BY s.store_name, fo.service_type;

In [0]:
%sql
select * from `03_gold_catalog`.gold_kpi.v_avg_days_in_shop

### Survey Coverage

In [0]:
%sql
CREATE OR REPLACE VIEW gold.v_survey_coverage AS
SELECT
    s.store_name,
    COUNT(*) AS surveys_sent,
    SUM(CASE WHEN responded_flag = 1 THEN 1 ELSE 0 END) AS surveys_responded,
    ROUND(SUM(CASE WHEN responded_flag = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS response_rate
FROM gold.fact_survey_final fs
JOIN gold.dim_store s ON fs.store_key = s.store_key
GROUP BY s.store_name;

### Survey Score Summary

In [0]:
%sql
CREATE OR REPLACE VIEW gold.v_survey_scores AS
SELECT
    s.store_name,
    ROUND(AVG(overall_satisfaction_rating), 2) AS avg_satisfaction,
    ROUND(AVG(cleanliness_rating), 2) AS avg_cleanliness,
    ROUND(AVG(communication_rating), 2) AS avg_communication,
    ROUND(AVG(work_quality_rating), 2) AS avg_work_quality,
    ROUND(AVG(delivered_on_time_rating), 2) AS avg_on_time,
    DENSE_RANK() OVER (ORDER BY AVG(overall_satisfaction_rating) DESC) AS satisfaction_rank
FROM gold.fact_survey_final fs
JOIN gold.dim_store s ON fs.store_key = s.store_key
WHERE responded_flag = 1
GROUP BY s.store_name;

### Revenue vs Budget


In [0]:
%sql
CREATE OR REPLACE VIEW gold.v_revenue_vs_budget AS
SELECT
    s.manager_name,
    b.budget_year,
    b.budget_month,
    b.budget_amount,
    SUM(fi.invoice_amount) AS actual_revenue,
    ROUND(SUM(fi.invoice_amount) / b.budget_amount, 2) AS achievement_ratio,
    DENSE_RANK() OVER (ORDER BY SUM(fi.invoice_amount) / b.budget_amount DESC) AS manager_rank
FROM gold.fact_invoice_final fi
JOIN gold.dim_store s ON fi.store_key = s.store_key
JOIN silver.ns_budget_silver b ON s.store_id = b.ns_store_id
    AND b.budget_month = MONTH(fi.invoice_date_key)
    AND b.budget_year = YEAR(fi.invoice_date_key)
GROUP BY s.manager_name, b.budget_year, b.budget_month, b.budget_amount;

### Top Technicians by Completion Time Accuracy

In [0]:
%sql
CREATE OR REPLACE VIEW gold.v_top_technicians_accuracy AS
SELECT
    s.store_name,
    t.technician_name,
    AVG(fo.planned_vs_actual_completion_days) AS avg_completion_variance,
    DENSE_RANK() OVER (
        PARTITION BY s.store_name ORDER BY AVG(fo.planned_vs_actual_completion_days)
    ) AS accuracy_rank
FROM gold.fact_order_cycle_final fo
JOIN gold.dim_store s ON fo.store_key = s.store_key
JOIN gold.dim_technician t ON fo.technician_key = t.technician_key
GROUP BY s.store_name, t.technician_name
QUALIFY accuracy_rank <= 5;

### Year‑to‑Date Revenue Growth 

In [0]:
%sql
CREATE OR REPLACE VIEW gold.v_ytd_revenue_growth AS
SELECT
    s.store_name,
    SUM(CASE WHEN YEAR(fi.invoice_date_key) = YEAR(CURRENT_DATE()) THEN fi.invoice_amount END)
        AS current_ytd_revenue,
    SUM(CASE WHEN YEAR(fi.invoice_date_key) = YEAR(CURRENT_DATE()) - 1 THEN fi.invoice_amount END)
        AS last_ytd_revenue,
    ROUND(
        (
            SUM(CASE WHEN YEAR(fi.invoice_date_key) = YEAR(CURRENT_DATE()) THEN fi.invoice_amount END) -
            SUM(CASE WHEN YEAR(fi.invoice_date_key) = YEAR(CURRENT_DATE()) - 1 THEN fi.invoice_amount END)
        ) / 
        SUM(CASE WHEN YEAR(fi.invoice_date_key) = YEAR(CURRENT_DATE()) - 1 THEN fi.invoice_amount END),
        2
    ) AS ytd_growth_rate,
    DENSE_RANK() OVER (
        ORDER BY (
            SUM(CASE WHEN YEAR(fi.invoice_date_key) = YEAR(CURRENT_DATE()) THEN fi.invoice_amount END) -
            SUM(CASE WHEN YEAR(fi.invoice_date_key) = YEAR(CURRENT_DATE()) - 1 THEN fi.invoice_amount END)
        ) DESC
    ) AS growth_rank
FROM gold.fact_invoice_final fi
JOIN gold.dim_store s ON fi.store_key = s.store_key
GROUP BY s.store_name;


### Stage‑wise Cycle Time

In [0]:
%sql
CREATE OR REPLACE VIEW gold.v_stage_cycle_time AS
SELECT
    s.store_name,
    fo.service_type,
    AVG(fo.days_vehicle_in_to_work_start) AS avg_vehicle_in_to_start,
    AVG(fo.days_work_start_to_completion) AS avg_work_start_to_completion,
    AVG(fo.days_completion_to_delivery) AS avg_completion_to_delivery
FROM gold.fact_order_cycle_final fo
JOIN gold.dim_store s ON fo.store_key = s.store_key
GROUP BY s.store_name, fo.service_type;

###  Estimator Accuracy

In [0]:
%sql
CREATE OR REPLACE VIEW gold.v_estimator_accuracy AS
WITH estimate_summary AS (
    SELECT
        e.order_id,
        e.estimator_id,
        MIN(estimate_amount) AS initial_estimate,
        MAX(estimate_amount) AS final_estimate
    FROM silver.estimate_silver e
    GROUP BY e.order_id, e.estimator_id
)
SELECT
    d.estimator_name,
    (final_estimate - initial_estimate) AS estimate_variance,
    ROUND((final_estimate - initial_estimate) / initial_estimate, 2) AS variance_pct,
    DENSE_RANK() OVER (ORDER BY ROUND((final_estimate - initial_estimate) / initial_estimate, 2)) AS estimator_rank
FROM estimate_summary es
JOIN gold.dim_estimator d ON es.estimator_id = d.estimator_id;

### Technician Workload

In [0]:
%sql
CREATE OR REPLACE VIEW gold.v_technician_workload AS
SELECT
    s.store_name,
    t.technician_name,
    COUNT(*) AS orders_handled,
    SUM(fo.total_days_in_shop) AS total_days_in_shop,
    DENSE_RANK() OVER (PARTITION BY s.store_name ORDER BY COUNT(*) DESC) AS workload_rank
FROM gold.fact_order_cycle_final fo
JOIN gold.dim_store s ON fo.store_key = s.store_key
JOIN gold.dim_technician t ON fo.technician_key = t.technician_key
GROUP BY s.store_name, t.technician_name;